In [1]:
# Load environment variables and verify the project setup.
import sys
from pathlib import Path

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ chromadb  — installed (1.5.9)
  ✓ langchain  — installed (1.3.2)
  ✓ langchain-chroma  — installed (1.1.0)
  ✓ langchain-community  — installed (0.4.2)
  ✓ langchain-core  — installed (1.4.0)
  ✓ langchain-experimental  — installed (0.4.2)
  ✓ langchain-openai  — installed (1.2.2)
  ✓ lxml  — installed (6.1.1)
  ✓ onnxruntime  — installed (1.19.2)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ rank-bm25  — installed (0.2.2)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# BM25 Retrieval

BM25 is a **sparse / lexical** retriever: it ranks documents by term-frequency statistics (TF-IDF with length normalization), matching the *exact words* in the query. No embeddings or API calls — it's fast, free, and strong at keyword/rare-term matching (IDs, code, names), where dense vector search can be weak.

It's the sparse half of [hybrid search](../hybrid-search/), which fuses BM25 with dense retrieval.

## 1. Load & chunk (semantic)

Chunk with `SemanticChunker` (OpenAI embeddings) so each chunk is topically coherent before indexing. Note: embeddings are used **here, for chunking** — the BM25 retriever itself stays purely lexical.

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

pdf_path = ROOT / "assets/sample-docs/sdlc-end-to-end.pdf"
docs = PyPDFLoader(str(pdf_path)).load()

# Semantic chunking: split at points where embedding similarity drops.
splitter = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",
)
chunks = splitter.split_documents(docs)
print(f"{len(docs)} pages -> {len(chunks)} semantic chunks")

/var/folders/gw/4x1sk2q13bz66qbs2jk5lpm80000gn/T/ipykernel_63741/2650433014.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/var/folders/gw/4x1sk2q13bz66qbs2jk5lpm80000gn/T/ipykernel_63741/2650433014.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


12 pages -> 23 semantic chunks


## 2. Build the BM25 retriever

`BM25Retriever` builds an in-memory index over the chunks (no vector store needed).

In [3]:
from langchain_community.retrievers import BM25Retriever

retriever = BM25Retriever.from_documents(chunks)
retriever.k = 3        # number of results to return
print("BM25 index built over", len(chunks), "chunks")

BM25 index built over 23 chunks


## 3. Query

BM25 shines on exact-term queries — here a query using wording straight from the doc.

In [4]:
query = "rollback plan and deployment runbook"
results = retriever.invoke(query)

for i, d in enumerate(results, 1):
    print(f"[{i}] page {d.metadata.get('page')}: {d.page_content[:140].strip()}\n")

[1] page 1: In Agile they are produced iteratively per sprint/epic rather than as a single
linear pass, but the logical dependency order remains the sam

[2] page 9: 6.1 CI/CD Pipeline
Purpose. Automated pipeline to build, test, and deliver/deploy software continuously and repeatably. Owner: DevOps / Plat

[3] page 6: Organization/project-level approach defining how testing is conducted overall. Owner: QA Manager  |  Input: SRS, risk profile  |  Flows to:

